In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
# # data = pd.read_csv("/content/sample_data/mnist_test.csv")
# # data2 = pd.read_csv("/content/sample_data/mnist_train_small.csv")

# data = pd.read_csv("Sensorless_drive_diagnosis.txt")

In [25]:
# Load raw dataset
sensorless_raw_df = pd.read_csv(
    "/content/Sensorless_drive_diagnosis.txt",
    sep=r"\s+",
    header=None
)

# Separate features and target
sensorless_features = sensorless_raw_df.iloc[:, :-1]
sensorless_target = sensorless_raw_df.iloc[:, -1]

print("Features shape:", sensorless_features.shape)
print("Target shape:", sensorless_target.shape)


Features shape: (58509, 48)
Target shape: (58509,)


In [26]:

# Train + temporary split
X_train_sensorless, X_temp_sensorless, y_train_sensorless, y_temp_sensorless = train_test_split(
    sensorless_features,
    sensorless_target,
    test_size=0.30,
    stratify=sensorless_target,
    random_state=42
)

# Validation + test split
X_val_sensorless, X_test_sensorless, y_val_sensorless, y_test_sensorless = train_test_split(
    X_temp_sensorless,
    y_temp_sensorless,
    test_size=0.50,
    stratify=y_temp_sensorless,
    random_state=42
)

print(X_train_sensorless.shape, X_val_sensorless.shape, X_test_sensorless.shape)


(40956, 48) (8776, 48) (8777, 48)


In [27]:

sensorless_scaler = StandardScaler()

X_train_sensorless = sensorless_scaler.fit_transform(X_train_sensorless)
X_val_sensorless   = sensorless_scaler.transform(X_val_sensorless)
X_test_sensorless  = sensorless_scaler.transform(X_test_sensorless)


In [28]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

sensorless_label_encoder = LabelEncoder()

y_train_encoded = sensorless_label_encoder.fit_transform(y_train_sensorless)
y_val_encoded   = sensorless_label_encoder.transform(y_val_sensorless)
y_test_encoded  = sensorless_label_encoder.transform(y_test_sensorless)

sensorless_num_classes = len(set(y_train_encoded))

y_train_categorical = to_categorical(y_train_encoded, sensorless_num_classes)
y_val_categorical   = to_categorical(y_val_encoded, sensorless_num_classes)
y_test_categorical  = to_categorical(y_test_encoded, sensorless_num_classes)


In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

sensorless_fnn_model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_sensorless.shape[1],)),
    Dropout(0.3),

    Dense(64, activation="relu"),
    Dropout(0.3),

    Dense(sensorless_num_classes, activation="softmax")
])

sensorless_fnn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

sensorless_fnn_model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 11)             │           715 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,243 (59.54 KB)

 Trainable params: 15,243 (59.54 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
sensorless_training_history = sensorless_fnn_model.fit(
    X_train_sensorless,
    y_train_categorical,
    validation_data=(X_val_sensorless, y_val_categorical),
    epochs=10,
    batch_size=256,
    verbose=1
)


Epoch 1/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9005 - loss: 0.2727 - val_accuracy: 0.9534 - val_loss: 0.1532
Epoch 2/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9111 - loss: 0.2440 - val_accuracy: 0.9594 - val_loss: 0.1427
Epoch 3/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9180 - loss: 0.2251 - val_accuracy: 0.9610 - val_loss: 0.1335
Epoch 4/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9228 - loss: 0.2125 - val_accuracy: 0.9646 - val_loss: 0.1234
Epoch 5/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9303 - loss: 0.1960 - val_accuracy: 0.9671 - val_loss: 0.1163
Epoch 6/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9352 - loss: 0.1856 - val_accuracy: 0.9697 - val_loss: 0.1082
Epoch 7/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9383 - loss: 0.1749 - val_accuracy: 0.9693 - val_loss: 0.1045
Epoch 8/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9421 - loss: 0.1651 - val_accuracy:

In [32]:
test_loss, test_accuracy = sensorless_fnn_model.evaluate(
    X_test_sensorless,
    y_test_categorical,
    verbose=0
)

print("Sensorless Drive Diagnosis Test Loss:", test_loss)
print("Sensorless Drive Diagnosis Test Accuracy:", test_accuracy)


Sensorless Drive Diagnosis Test Loss: 0.10887176543474197
Sensorless Drive Diagnosis Test Accuracy: 0.9741369485855103
